# Session 05 - Feature Engineering Fundamentals

This notebook engineers features for a synthetic workplace churn dataset and compares a baseline feature set against an engineered feature set.

**Learning outcome:** design features that improve model learnability while preserving evaluation integrity.

**Professional rule:** split before model fitting, exclude future/target-derived columns, and document why each engineered feature is safe.

In [15]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

## 1. Load the dataset
The dataset contains safe pre-prediction fields plus intentionally leaky post-outcome fields for exclusion practice.

In [16]:
DATA_PATH = 'workplace_churn_feature_engineering_dataset.csv'
df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

(720, 21)


,customer_id,signup_date,last_login_date,region,sector,plan_type,contract_type,acquisition_channel,monthly_fee_gbp,tenure_months,...,active_days_30d,support_tickets_90d,invoices_late_6m,satisfaction_score,renewal_month,account_notes,cancellation_reason,account_closed_date,retention_offer_after_churn,churned
0,CUST-1001,2022-09-21,2024-12-29,Midlands,Education,Basic,Monthly,Sales-led,55.75,27,...,5,1,0,4.9,4,Healthy adoption in team; users asked for repo...,NaN,NaN,NaN,0
1,CUST-1002,2021-02-22,2024-12-20,North West,Healthcare,Basic,Monthly,Sales-led,49.21,46,...,9,3,0,6.8,3,Invoice query raised; finance contact requeste...,NaN,NaN,NaN,0
2,CUST-1003,2021-01-21,2024-12-31,Northern Ireland,Professional Services,Basic,Monthly,Self-serve,36.68,47,...,8,1,4,7.1,7,Multiple support contacts; wants faster respon...,business_closed,2025-02-01,none,1
3,CUST-1004,2020-07-08,2024-12-10,Wales,Technology,Basic,Monthly,Sales-led,43.12,53,...,0,2,1,7.2,3,Support issue around onboarding and admin setup.,support,2025-02-07,success_call,1
4,CUST-1005,2020-01-07,2024-12-24,London,Professional Services,Basic,Annual,Self-serve,34.46,59,...,12,2,0,7.7,2,Healthy adoption in team; users asked for repo...,NaN,NaN,NaN,0


## 2. Inspect target balance and columns
Before engineering features, understand the target and identify columns that must not be used as inputs.

In [17]:
target = 'churned'
print(df[target].value_counts(normalize=True).rename('target_rate'))
print('\nColumns:')
print(df.columns.tolist())

churned
0    0.747222
1    0.252778
Name: target_rate, dtype: float64

Columns:
['customer_id', 'signup_date', 'last_login_date', 'region', 'sector', 'plan_type', 'contract_type', 'acquisition_channel', 'monthly_fee_gbp', 'tenure_months', 'usage_minutes_30d', 'active_days_30d', 'support_tickets_90d', 'invoices_late_6m', 'satisfaction_score', 'renewal_month', 'account_notes', 'cancellation_reason', 'account_closed_date', 'retention_offer_after_churn', 'churned']


## 3. Define leaky columns and safe baseline features
Leaky columns are not available at the prediction moment or encode the answer after churn has happened.

In [18]:
leaky_columns = ['cancellation_reason', 'account_closed_date', 'retention_offer_after_churn']
identifier_columns = ['customer_id']

baseline_numeric = [
    'monthly_fee_gbp', 'tenure_months', 'usage_minutes_30d', 'active_days_30d',
    'support_tickets_90d', 'invoices_late_6m', 'satisfaction_score', 'renewal_month'
]
baseline_categorical = ['region', 'sector', 'plan_type', 'contract_type', 'acquisition_channel']

X_base = df[baseline_numeric + baseline_categorical].copy()
y = df[target].copy()

print('Excluded as leaky:', leaky_columns)
print('Baseline columns:', X_base.columns.tolist())

Excluded as leaky: ['cancellation_reason', 'account_closed_date', 'retention_offer_after_churn']
Baseline columns: ['monthly_fee_gbp', 'tenure_months', 'usage_minutes_30d', 'active_days_30d', 'support_tickets_90d', 'invoices_late_6m', 'satisfaction_score', 'renewal_month', 'region', 'sector', 'plan_type', 'contract_type', 'acquisition_channel']


## 4. Create the same train/test split for all comparisons
This keeps the comparison fair. Only the feature set changes.

In [19]:
X_train_base, X_test_base, y_train, y_test = train_test_split(
    X_base, y, test_size=0.25, random_state=42, stratify=y
)
print(X_train_base.shape, X_test_base.shape)

(540, 13) (180, 13)


## 5. Build a reusable modelling function
The model is intentionally simple. The session is about feature engineering, not advanced model selection.

In [20]:
def build_model(numeric_cols, categorical_cols):
    numeric_pipe = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])
    categorical_pipe = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ])
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numeric_pipe, numeric_cols),
            ('cat', categorical_pipe, categorical_cols)
        ]
    )
    model = LogisticRegression(max_iter=1000)
    return Pipeline(steps=[('preprocess', preprocessor), ('model', model)])

def evaluate_pipeline(name, pipe, X_train, X_test, y_train, y_test):
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    proba = pipe.predict_proba(X_test)[:, 1]
    return {
        'input_set': name,
        'accuracy': round(accuracy_score(y_test, pred), 3),
        'precision': round(precision_score(y_test, pred, zero_division=0), 3),
        'recall': round(recall_score(y_test, pred, zero_division=0), 3),
        'f1': round(f1_score(y_test, pred, zero_division=0), 3),
        'roc_auc': round(roc_auc_score(y_test, proba), 3),
        'n_features_before_encoding': X_train.shape[1]
    }

## 6. Baseline model
Train a simple model using only raw safe input columns.

In [21]:
baseline_pipe = build_model(baseline_numeric, baseline_categorical)
baseline_result = evaluate_pipeline('baseline_safe_raw_features', baseline_pipe, X_train_base, X_test_base, y_train, y_test)
baseline_result

{'input_set': 'baseline_safe_raw_features',
 'accuracy': 0.794,
 'precision': 0.618,
 'recall': 0.467,
 'f1': 0.532,
 'roc_auc': np.float64(0.82),
 'n_features_before_encoding': 13}

## 7. Engineer features with Pandas
Feature engineering converts raw fields into more learnable signals. Each feature must be checked for leakage.

In [22]:
def add_engineered_features(data):
    out = data.copy()
    # Date features - derived from dates known before prediction.
    signup = pd.to_datetime(out['signup_date'], errors='coerce')
    last_login = pd.to_datetime(out['last_login_date'], errors='coerce')
    prediction_date = pd.to_datetime('2024-12-31')
    out['signup_month'] = signup.dt.month
    out['signup_quarter'] = signup.dt.quarter
    out['days_since_last_login'] = (prediction_date - last_login).dt.days

    # Binning - convert continuous tenure into lifecycle groups.
    out['tenure_band'] = pd.cut(
        out['tenure_months'],
        bins=[0, 6, 18, 36, 999],
        labels=['new', 'growing', 'established', 'long_term']
    ).astype('object')

    # Transformations and ratios.
    out['usage_per_active_day'] = out['usage_minutes_30d'] / out['active_days_30d'].replace(0, np.nan)
    out['fee_per_usage_minute'] = out['monthly_fee_gbp'] / out['usage_minutes_30d'].replace(0, np.nan)
    out['ticket_rate_per_tenure'] = out['support_tickets_90d'] / out['tenure_months'].replace(0, np.nan)

    # Interaction-style features.
    out['low_usage_high_tickets_flag'] = ((out['usage_minutes_30d'] < 120) & (out['support_tickets_90d'] >= 3)).astype(int)
    out['monthly_contract_low_satisfaction_flag'] = ((out['contract_type'] == 'Monthly') & (out['satisfaction_score'] < 6)).astype(int)

    # Text-derived features from account notes.
    notes = out['account_notes'].fillna('').str.lower()
    out['account_note_length'] = notes.str.len()
    out['mentions_price_issue'] = notes.str.contains('price|invoice|fee|discount', regex=True).astype(int)
    out['mentions_support_issue'] = notes.str.contains('support|onboarding|response', regex=True).astype(int)
    return out

df_eng = add_engineered_features(df)
df_eng.filter(regex='tenure_band|usage_per|fee_per|ticket_rate|flag|note|mentions|signup_|days_since').head()

,signup_date,account_notes,signup_month,signup_quarter,days_since_last_login,tenure_band,usage_per_active_day,fee_per_usage_minute,ticket_rate_per_tenure,low_usage_high_tickets_flag,monthly_contract_low_satisfaction_flag,account_note_length,mentions_price_issue,mentions_support_issue
0,2022-09-21,Healthy adoption in team; users asked for repo...,9,3,2,established,22.400000,0.497768,0.037037,0,1,65,0,0
1,2021-02-22,Invoice query raised; finance contact requeste...,2,1,11,long_term,32.633333,0.167552,0.065217,0,0,62,1,0
2,2021-01-21,Multiple support contacts; wants faster respon...,1,1,0,long_term,15.350000,0.298697,0.021277,0,0,54,0,1
3,2020-07-08,Support issue around onboarding and admin setup.,7,3,21,long_term,NaN,0.249682,0.037736,0,0,48,0,1
4,2020-01-07,Healthy adoption in team; users asked for repo...,1,1,7,long_term,21.150000,0.135776,0.033898,0,0,65,0,0


## 8. Compare baseline vs engineered feature set
Use the same target, same test size, same random state, same model type, and same metric set.

In [26]:
engineered_numeric = baseline_numeric + [
    'signup_month', 'signup_quarter', 'days_since_last_login',
    'usage_per_active_day', 'fee_per_usage_minute', 'ticket_rate_per_tenure',
    'low_usage_high_tickets_flag', 'monthly_contract_low_satisfaction_flag',
    'account_note_length', 'mentions_price_issue', 'mentions_support_issue', 'late_payment_rate', 'days_to_renewal', 'satisfaction_x_support_load'
]
engineered_categorical = baseline_categorical + ['tenure_band']

X_eng = df_eng[engineered_numeric + engineered_categorical].copy()
X_train_eng, X_test_eng, _, _ = train_test_split(
    X_eng, y, test_size=0.25, random_state=42, stratify=y
)

engineered_pipe = build_model(engineered_numeric, engineered_categorical)
engineered_result = evaluate_pipeline('engineered_features', engineered_pipe, X_train_eng, X_test_eng, y_train, y_test)

results = pd.DataFrame([baseline_result, engineered_result])
results

,input_set,accuracy,precision,recall,f1,roc_auc,n_features_before_encoding
0,baseline_safe_raw_features,0.794,0.618,0.467,0.532,0.82,13
1,engineered_features,0.750,0.500,0.356,0.416,0.78,28


## 9. Document engineered features
A professional feature is not just code. It needs rationale, leakage status, and a caveat.

In [27]:
feature_summary = pd.DataFrame([
    {'feature':'tenure_band', 'type':'binning', 'rationale':'Captures customer lifecycle stage.', 'leakage_check':'Uses signup/tenure known before prediction.', 'caveat':'Bins may hide differences inside a band.'},
    {'feature':'usage_per_active_day', 'type':'ratio', 'rationale':'Measures usage intensity, not just total usage.', 'leakage_check':'Uses recent pre-prediction usage window.', 'caveat':'Missing or zero active days need careful handling.'},
    {'feature':'low_usage_high_tickets_flag', 'type':'interaction', 'rationale':'Combines weak adoption and support friction.', 'leakage_check':'Uses pre-prediction tickets and usage.', 'caveat':'May reflect service-quality issues, not customer intent.'},
    {'feature':'mentions_price_issue', 'type':'text-derived flag', 'rationale':'Captures price concern from account notes.', 'leakage_check':'Only safe if note exists before prediction.', 'caveat':'Manual notes can be inconsistent or biased.'},
])
feature_summary

,feature,type,rationale,leakage_check,caveat
0,tenure_band,binning,Captures customer lifecycle stage.,Uses signup/tenure known before prediction.,Bins may hide differences inside a band.
1,usage_per_active_day,ratio,"Measures usage intensity, not just total usage.",Uses recent pre-prediction usage window.,Missing or zero active days need careful handl...
2,low_usage_high_tickets_flag,interaction,Combines weak adoption and support friction.,Uses pre-prediction tickets and usage.,"May reflect service-quality issues, not custom..."
3,mentions_price_issue,text-derived flag,Captures price concern from account notes.,Only safe if note exists before prediction.,Manual notes can be inconsistent or biased.


## 10. Save outputs
These outputs can be placed in a GitHub repository with the notebook and report.

In [28]:
results.to_csv('baseline_vs_engineered_results.csv', index=False)
feature_summary.to_csv('engineered_feature_summary.csv', index=False)
df_eng.to_csv('workplace_churn_engineered_features.csv', index=False)
print('Saved results, feature summary, and engineered dataset.')

Saved results, feature summary, and engineered dataset.


## Student exercise
Create three additional engineered features. For each one, write:

1. Feature name.
2. Business rationale.
3. Why it may help a model.
4. Leakage check.
5. Possible bias or governance caveat.

In [25]:
# Student exercise - three additional engineered features

# Feature 1: Late payment rate (ratio)
df_eng['late_payment_rate'] = df_eng['invoices_late_6m'] / df_eng['tenure_months'].replace(0, np.nan)

# Feature 2: Days to renewal (date-derived)
prediction_date = pd.to_datetime('2024-12-31')
df_eng['days_to_renewal'] = df_eng['renewal_month'].apply(
    lambda m: ((pd.Timestamp(2025, int(m), 1) - prediction_date).days) % 365
)

# Feature 3: Satisfaction x support load (interaction)
df_eng['satisfaction_x_support_load'] = (
    (10 - df_eng['satisfaction_score'].fillna(5)) * df_eng['support_tickets_90d']
)

## Feature 1 — late_payment_rate

1. Late payment rate

2. A customer with 2 late invoices after 3 months is a much bigger concern than one with 2 late invoices after 3 years

3. Normalising by tenure separates payment friction from relationship length, giving the model a cleaner signal

4. Both invoices_late_6m and tenure_months are known before prediction — no leakage

5. New customers have very low tenure, which inflates the ratio — consider clipping at a sensible maximum

## Feature 2 — days_to_renewal

1. Days to renewal

2. Churn risk spikes around renewal time; knowing how close a customer is to their next renewal date adds a temporal signal

3. Customers approaching renewal are in a decision window — this helps the model weight those cases higher

4. renewal_month is recorded before prediction — no leakage

5. Works best for annual contracts; monthly-contract customers have less meaningful renewal cycles

## Feature 3 — satisfaction_x_support_load

1. Satisfaction × support load interaction

2. High ticket volume from a satisfied customer is fine; high tickets from a dissatisfied one is a red flag — this feature captures that combination

3. Interaction terms let a linear model learn joint effects it otherwise can't detect

4. Both satisfaction_score and support_tickets_90d are pre-prediction — no leakage

5. Missing satisfaction scores are filled with 5 (neutral) — this assumption should be documented and could be changed

## Feature Engineering Report
Three additional features were engineered from the workplace churn dataset to improve the model's ability to identify customers at risk of leaving.

**Feature 1: late_payment_rate**

This feature divides the number of late invoices in the last six months by the customer's tenure in months. The business rationale is that payment friction means something very different depending on how long a customer has been with the company — two late payments in three months is far more concerning than two late payments over three years. This ratio gives the model a cleaner signal of payment behaviour relative to relationship length. The feature is safe from leakage as both invoices_late_6m and tenure_months are known before the prediction date. A governance caveat is that new customers have very low tenure values which can inflate the ratio significantly, so clipping the output at a sensible maximum should be considered.

**Feature 2: days_to_renewal**

This feature calculates how many days remain until a customer's next renewal date. Customers are most likely to churn at the point when their contract is up for renewal, as this is the natural decision moment. Giving the model visibility of how close each customer is to that window allows it to weight at-risk customers more appropriately. The renewal_month field is recorded in the system before prediction and is therefore leakage-free. The main caveat is that this feature is most meaningful for annual contracts. Monthly contract customers face a renewal decision every month, which reduces the signal value.

**Feature 3: satisfaction_x_support_load**

This interaction feature multiplies an inverted satisfaction score by the number of support tickets raised in the last 90 days. A high volume of support tickets alone does not necessarily indicate churn risk, as engaged, satisfied customers can be heavy support users. However, high ticket volume combined with low satisfaction is a strong warning sign. Combining them into a single term allows a logistic regression model to capture this joint effect without requiring a more complex model. Both inputs are pre-prediction so there is no leakage risk. The key caveat is that missing satisfaction scores are filled with a neutral value of 5, which is an assumption that should be revisited if the missingness is not random.